# ComplianceGPT — Single Run

This notebook runs the pipeline **in-process** (recommended) so you always use the exact `pipeline.py` you edited.
If you prefer `!python pipeline.py ...`, use the CLI-enabled pipeline and pass `--ccs_path` explicitly.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Cell 2: deps (minimal, reproducible)
!pip -q install sentence-transformers faiss-cpu rank-bm25 accelerate

import torch
if torch.cuda.is_available():
    !pip -q install bitsandbytes
print("cuda:", torch.cuda.is_available(), "torch:", torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 46.4 MB/s eta 0:00:00
cuda: True torch: 2.9.0+cu126


In [3]:
# Cell 3: paths + import sanity
from pathlib import Path
import sys

PIPELINE_DIR = "/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/pipeline"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

REV5_CATALOG_PATH = "/content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = "/content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"
assert Path(PIPELINE_DIR).exists(), "PIPELINE_DIR missing"
assert Path(REV5_CATALOG_PATH).exists(), "REV5_CATALOG_PATH missing"

import pipeline, retriever_s7
print("pipeline:", pipeline.__file__)
print("retriever_s7:", retriever_s7.__file__)


[DEBUG] Pipeline added paths: ['/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/pipeline', '/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/generator', '/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/generator/verifier', '/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/QUR_generator', '/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/retriever', '/content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt']
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib
pipeline: /content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/pipeline/pipeline.py
retriever_s7: /content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/retriever/retriever_s7.py


In [4]:
import retriever_s7, inspect

print("retriever_s7 file:", retriever_s7.__file__)
src = inspect.getsource(retriever_s7.ComplianceGPTRetriever.__init__)
print("has clause-id ids patch:", "clause_id" in src and "self.ids" in src)
print("---- snippet ----")
print("\n".join(src.splitlines()[-15:]))


retriever_s7 file: /content/drive/MyDrive/ComplianceGPT_v2/src/compliancegpt/retriever/retriever_s7.py
has clause-id ids patch: True
---- snippet ----
            if ctl:
                self.control_to_clause_ids[ctl].append(r["id"])

        # Build BM25 on control-level aggregated docs
        ctl_docs = build_control_docs_from_clauses(self.records)
        self.bm25, self.bm25_control_ids = build_bm25(ctl_docs, k1=config.bm25_k1, b=config.bm25_b)

        # Dense clause index + adapter
        self.dense_adapter, self.clause_id_to_text, self.clause_to_control = build_dense_retriever(self.records, dense_model_id=config.dense_model_id)

        # Reranker
        self.reranker = build_reranker(config.reranker_model_id)

        # Fallback control text for reranker
        self.id_to_text_map = build_control_fallback_text_map(self.records)


In [5]:
r = retriever_s7.ComplianceGPTRetriever(
    catalog_path=REV5_CATALOG_PATH,
    config=retriever_s7.RetrievalConfig()
)
print("len(r.ids) =", len(r.ids))
print("sample ids =", r.ids[:5])
print("has _smt in first 500 =", any("_smt" in x for x in r.ids[:500]))
print("has _gdn in first 500 =", any("_gdn" in x for x in r.ids[:500]))


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

len(r.ids) = 3136
sample ids = ['ac-1_smt', 'ac-1_smt.a', 'ac-1_smt.a.1', 'ac-1_smt.a.1.a', 'ac-1_smt.a.1.b']
has _smt in first 500 = True
has _gdn in first 500 = True


In [6]:
# Cell 4: run rev5 PRESERVE question
from pipeline import ComplianceGPTPipeline

pipe = ComplianceGPTPipeline(
    framework_version="rev5",
    ccs_path=REV5_CATALOG_PATH,
    strict_ccs_assert=True,
)

q = "When must account managers and the designated personnel be notified about user account changes?"
out = pipe.answer(q, top_k=10)

print("status:", out["contract"]["status"])
print("selected_source_ids:", out["contract"].get("selected_source_ids"))
print("odp_required_list:", out["contract"].get("odp_required_list"))
if out.get("docs"):
    print("top1:", out["docs"][0].get("id"), out["docs"][0].get("control_id"))


[System] Loading model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[CCS] Loaded corpus from: /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl
[CCS] docs=3136 kind_counts={'smt': 2125, 'gdn': 1011, 'obj': 0, 'other': 0}


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


status: PARAMS_REQUIRED
selected_source_ids: ['ac-2.1_gdn', 'ac-2_smt.h.3']
odp_required_list: ['ac-02_odp.08']
top1: ac-2_smt.h.3 AC-2


In [7]:
# Cell 5: run rev4 PRESERVE question
from pipeline import ComplianceGPTPipeline

pipe = ComplianceGPTPipeline(
    framework_version="rev4",
    ccs_path=REV4_CATALOG_PATH,
    strict_ccs_assert=True,
)

q = "When must account managers and the designated personnel be notified about user account changes?"
out = pipe.answer(q, top_k=10)

print("status:", out["contract"]["status"])
print("selected_source_ids:", out["contract"].get("selected_source_ids"))
print("odp_required_list:", out["contract"].get("odp_required_list"))
if out.get("docs"):
    print("top1:", out["docs"][0].get("id"), out["docs"][0].get("control_id"))


[System] Loading model: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[Pipeline] QUR initialized.
[Pipeline] Retriever loading: /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[CCS] Loaded corpus from: /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl
[CCS] docs=2281 kind_counts={'smt': 1586, 'gdn': 695, 'obj': 0, 'other': 0}
status: PARAMS_REQUIRED
selected_source_ids: ['ac-9.3_smt']
odp_required_list: ['ac-9.3_prm_1', 'ac-9.3_prm_2']
top1: ac-9.3_smt AC-9.3


In [8]:
# Cell 6: run a nonsense query
from pipeline import ComplianceGPTPipeline

pipe = ComplianceGPTPipeline(
    framework_version="rev5",
    ccs_path=REV5_CATALOG_PATH,
    strict_ccs_assert=True,
)

q = "zzzxqv not a real control question?"
out = pipe.answer(q, top_k=10)

print("status:", out["contract"]["status"])
print("selected_source_ids:", out["contract"].get("selected_source_ids"))
print("odp_required_list:", out["contract"].get("odp_required_list"))
print("evidence_spans_len:", len(out["contract"].get("evidence_spans", [])))
print("answer_text_preview:", repr(out["contract"].get("answer_text", "")[:80]))
if out.get("docs"):
    print("top1:", out["docs"][0].get("id"), out["docs"][0].get("control_id"))


[System] Loading model: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[Pipeline] QUR initialized.
[Pipeline] Retriever loading: /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[CCS] Loaded corpus from: /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl
[CCS] docs=3136 kind_counts={'smt': 2125, 'gdn': 1011, 'obj': 0, 'other': 0}
status: NO_EVIDENCE
selected_source_ids: []
odp_required_list: []
top1: au-14.3_gdn AU-14.3
